# Directional Locomotion Tutorial

This notebook walks through the full pipeline for training a muscle-driven humanoid
to walk in any of 8 compass directions (N / NE / E / SE / S / SW / W / NW).

**Sections:**
1. Motion clip selection
2. Dataset collection (BC from teacher)
3. Policy architecture
4. Training
5. Evaluation (8 directions x 10 seeds x 200 steps)
6. Render: single agent, all 8 directions
7. Render: 1-vs-1 ChaseTag

**Prerequisites:** run from the repo root (`myosuite4/`).  
Install with `pip install -e .[dev]`.

## 0 — Imports and paths

In [ ]:
from __future__ import annotations

import math
import os
from pathlib import Path

import mujoco
import numpy as np

# Works whether the notebook is run from the repo root or from tutorials/
_here = Path(os.getcwd())
REPO_ROOT = _here if (_here / 'myosuite').exists() else _here.parent
DATA_NPZ  = REPO_ROOT / 'runs' / 'bc_directional_v2' / 'bc_directional.npz'
CKPT_DIR  = REPO_ROOT / 'runs' / 'bc_directional_v2'
# Use the best available checkpoint: bc_directional_v2 (350k
# transitions) is the recommended default -- it has the best
# closed-loop survival@200 (72.5%) of all runs tried, including the
# larger 360k-transition v3 run, which regressed to 50% despite a
# marginally lower validation loss (see the Evaluation section for
# why val loss alone is a poor proxy for survival). Fall back to v3,
# then the original v1 (60k), only if v2 isn't trained locally.
CKPT = next(
    (REPO_ROOT / 'runs' / d / 'policy_bc_best.pt'
     for d in ('bc_directional_v2', 'bc_directional_v3', 'bc_directional_v1')
     if (REPO_ROOT / 'runs' / d / 'policy_bc_best.pt').exists()),
    CKPT_DIR / 'policy_bc_best.pt',
)
OBS_DIM   = 528  # qpos_local(82)+qvel_local(82)+act(354)+root_vel_body(2)+heading_cmd(2)+orientation(6)
ACT_DIM   = 354
RENDER_DIR = REPO_ROOT / 'renders'
RENDER_DIR.mkdir(exist_ok=True)

print('Repo root:', REPO_ROOT)
print('Checkpoint:', CKPT)


---
## 1 — Motion clip selection

**Why circular clips?**  
A single clockwise or counter-clockwise walking circle sweeps through all compass
directions continuously. By picking the right *frame*, we can initialise the agent
already heading toward any target direction — no need for 8 separate straight-walk
clips.  
Two clips suffice to cover all 8 sectors:

| Clip | Subject | Sectors covered |
|------|---------|-----------------|
| `WalkInClockwiseCircle01` | 4 | NE, N, NW, W, S, SE |
| `WalkInCounterClockwiseCircle08` | 4 | **E**, **SW** |

Below we download both clips from HuggingFace, then for each of the 8 sectors
search every frame (step 5) for the smallest angular error.

In [ ]:
from huggingface_hub import hf_hub_download
from myosuite.core.trajectory_io import load_motion_clip
from myosuite.integrations.musclemimic.fullbody_model import (
    compile_mimic_fullbody_mjmodel,
    default_mimic_fullbody_config,
)

GAIT_REPO = 'amathislab/musclemimic-retargeted'
SUBJECT   = '4'

# Build a reference MuJoCo model to get nq / nv
cfg = default_mimic_fullbody_config()
model, _, _ = compile_mimic_fullbody_mjmodel(cfg)

CLIP_NAMES = [
    'WalkInClockwiseCircle01',
    'WalkInCounterClockwiseCircle08',
]

clips = {}
for name in CLIP_NAMES:
    hf_path = hf_hub_download(
        repo_id=GAIT_REPO,
        filename=f'MyoFullBody/gmr/KIT/{SUBJECT}/{name}_poses.npz',
        repo_type='dataset',
    )
    clips[name] = load_motion_clip(
        Path(hf_path),
        expected_nq=model.nq,
        expected_nv=model.nv,
    )
    print(f'{name}: {len(clips[name].qpos)} frames')
print('Clips loaded.')


In [ ]:
# For each of the 8 compass sectors compute which clip + frame has the smallest
# angular error to the target direction.

SECTOR_DEG = list(range(0, 360, 45))  # 0=E, 45=NE, ...
LABELS     = ['E', 'NE', 'N', 'NW', 'W', 'SW', 'S', 'SE']

_data_ref = mujoco.MjData(model)

def clip_walk_angle_deg(clip, frame: int) -> float:
    """Return the horizontal walking direction (degrees) at a given clip frame."""
    _data_ref.qpos[:] = clip.qpos[frame]
    _data_ref.qvel[:] = clip.qvel[frame]
    mujoco.mj_forward(model, _data_ref)
    vx, vy = float(_data_ref.qvel[0]), float(_data_ref.qvel[1])
    return math.degrees(math.atan2(vy, vx))

def angular_error_deg(a: float, b: float) -> float:
    """Signed shortest angular distance (degrees)."""
    return abs((a - b + 180) % 360 - 180)

print(f'{'Sector':>6}  {'Best clip':45}  {'Frame':>5}  {'Error deg':>9}')
sector_assignments = {}   # store for later use
for deg, label in zip(SECTOR_DEG, LABELS):
    best_err, best_clip, best_frame = float('inf'), '', 0
    for name, clip in clips.items():
        for frame in range(0, len(clip.qpos), 5):
            ang = clip_walk_angle_deg(clip, frame)
            err = angular_error_deg(ang, float(deg))
            if err < best_err:
                best_err, best_clip, best_frame = err, name, frame
    sector_assignments[label] = (best_clip, best_frame, best_err)
    print(f'{label:>6}  {best_clip:45}  {best_frame:>5}  {best_err:>9.2f}')


---
## 2 — Dataset collection (BC from teacher)

**Teacher policy:** `amathislab/mm-10m-2` — a pre-trained MuscleMimic full-body
locomotion policy.  We roll it out on the two circular clips and record real
`(obs, action)` pairs: the teacher's own muscle-activation action for every
step, no dummy/placeholder actions.

**Observation format (528 dims):**

| Slice | Content | Dim |
|-------|---------|-----|
| `[:82]` | `qpos[7:]` joint angles (excl. root free-joint) | 82 |
| `[82:164]` | `qvel[6:]` joint velocities (excl. root 6-DOF) | 82 |
| `[164:518]` | muscle activations `act` | 354 |
| `[518:520]` | root XY velocity in body frame | 2 |
| `[520:522]` | heading command `[cos(theta), sin(theta)]` | 2 |
| `[522:528]` | orientation: roll, pitch, wx\_body, wy\_body, wz\_world, vz | 6 |

**Body-frame velocity:** rotating `qvel[:2]` by negative pelvis yaw makes
'forward speed' heading-invariant — the agent sees the same value regardless
of which global direction it walks.

**Heading label from the clip itself:** because the circular clips
continuously sweep every compass direction, `heading_cmd` for each recorded
transition is set to the clip's *actual* instantaneous walking direction at
that frame (`atan2(qvel_y, qvel_x)`) — one continuous rollout yields balanced
coverage of all 8 sectors, no per-sector episodes needed.

**Dataset (`runs/bc_directional_v2/bc_directional.npz`):** 350,000 transitions
(175,000 per clip × 2 circular clips), collected by actually running the
teacher via
`myosuite.integrations.musclemimic.bc_directional_collector.collect_bc_dataset`
on a Linux host (Modal), since loading the teacher's Orbax checkpoint requires
`orbax-checkpoint`, which this repo excludes on macOS (see `pyproject.toml`'s
`sys_platform != 'darwin'` markers on the `mjx` extra). See
`scripts/modal_bc_directional_pipeline.py` for the exact launcher used.

The original tutorial design targeted 800,000 transitions (100k × 8
directions, collected directly per-sector rather than from continuous circular
rollouts). An earlier run (`bc_directional_v1`, kept for comparison in
`runs/bc_directional_v1/`) used a scaled-down 60k-transition dataset that
produced a working-but-weak policy (survival@200 = 6.2%, see below). This
run scales up to **350k transitions (44% of the original 800k target)** —
still short of the full design but enough to close most of the survival gap
(see the Evaluation section below).

**Dataset note:** the `heading_cmd` occupies `obs[520:522]` (after body-frame vel
at `[518:520]`). Earlier experiments used `[518:520]` for heading — this was a bug
that caused the agent to confuse velocity and command signals.

In [ ]:
from huggingface_hub import snapshot_download
from myosuite.integrations.musclemimic.fullbody_local_policy import (
    load_local_policy_artifacts,
    LocalPolicyRunner,
    FullbodyObsAdapter,
)
from scipy.spatial.transform import Rotation as R

# ---- helper: build the 528-dim directional obs ----

def build_directional_obs(
    data: mujoco.MjData,
    theta_rad: float,
) -> np.ndarray:
    """Build the 528-dim obs for the directional env from a MjData state.

    Obs layout:
        [0:82]    qpos[7:]  local joint angles
        [82:164]  qvel[6:]  local joint velocities
        [164:518] act       muscle activations
        [518:520] vel_body  root XY velocity in body frame
        [520:522] heading_cmd  [cos(theta), sin(theta)]
        [522:528] orientation  roll, pitch, wx_b, wy_b, wz_w, vz

    Args:
        data: Current simulation state (after mj_forward).
        theta_rad: Target heading in radians (0 = East).

    Returns:
        obs: float32 array of shape (528,).
    """
    qpos = data.qpos.astype(np.float32)
    qvel = data.qvel.astype(np.float32)
    act  = data.act.astype(np.float32)

    # Body-frame XY velocity: rotate global qvel[:2] by -pelvis_yaw
    pelvis_quat = qpos[3:7]                  # wxyz quaternion of root
    yaw = float(R.from_quat(pelvis_quat[[1, 2, 3, 0]]).as_euler('zyx')[0])
    c, s = math.cos(-yaw), math.sin(-yaw)
    vx_g, vy_g = float(qvel[0]), float(qvel[1])
    vel_body = np.array([c * vx_g - s * vy_g, s * vx_g + c * vy_g], dtype=np.float32)

    heading_cmd = np.array([math.cos(theta_rad), math.sin(theta_rad)], dtype=np.float32)

    # 6-dim orientation features
    w, x, y, z = pelvis_quat[0], pelvis_quat[1], pelvis_quat[2], pelvis_quat[3]
    roll  = math.atan2(2*(w*x + y*z), 1 - 2*(x*x + y*y))
    pitch = math.asin(float(np.clip(2*(w*y - z*x), -1.0, 1.0)))
    wx_w, wy_w, wz_w = float(qvel[3]), float(qvel[4]), float(qvel[5])
    wx_b = c * wx_w - s * wy_w
    wy_b = s * wx_w + c * wy_w
    vz   = float(qvel[2])
    orientation = np.array([roll, pitch, wx_b, wy_b, wz_w, vz], dtype=np.float32)

    return np.concatenate([
        qpos[7:],          # 82 dims  [0:82]
        qvel[6:],          # 82 dims  [82:164]
        act,               # 354 dims [164:518]
        vel_body,          # 2 dims   [518:520]
        heading_cmd,       # 2 dims   [520:522]
        orientation,       # 6 dims   [522:528]
    ])                     # total 528


print('build_directional_obs defined.')
print('obs_dim expected: 528')


In [ ]:
# ---- Real teacher-rollout data collection (350k transitions) ----
#
# NOTE ON PLATFORM: loading the teacher's Orbax checkpoint requires
# `orbax-checkpoint`, which this repo's pyproject.toml excludes on macOS
# (`sys_platform != 'darwin'` markers on the `mjx` extra, which is where
# orbax lives). If you're on macOS and DATA_NPZ doesn't already exist, this
# cell will raise ImportError -- run the collection on Linux instead, e.g.:
#
#     modal run --detach scripts/modal_bc_directional_pipeline.py \
#         --samples-per-clip 175000 --episode-len 400 --epochs 120 \
#         --run-name bc_directional_v2
#
# then download the resulting runs/bc_directional_v2/bc_directional.npz
# (`modal volume get myosuite-bc-directional-out bc_directional_v2/bc_directional.npz runs/bc_directional_v2/bc_directional.npz`)
# before re-running this cell. (A follow-up 360k-transition run,
# bc_directional_v3, was also tried -- see the Evaluation section --
# but is not recommended as the default; use `--samples-per-clip 175000`
# / `--run-name bc_directional_v2` unless you specifically want to
# reproduce that comparison.)

if DATA_NPZ.exists():
    print(f'Dataset already exists: {DATA_NPZ}')
    d = np.load(DATA_NPZ)
    print(f'  obs: {d["obs"].shape}  actions: {d["actions"].shape}')
else:
    from myosuite.integrations.musclemimic.bc_directional_collector import (
        BcCollectionConfig,
        collect_bc_dataset,
    )

    # Reuse the same circular clips loaded in Section 1 (`clips` dict, keyed
    # by clip name) -- one continuous rollout per clip sweeps every heading.
    teacher_root = Path(snapshot_download(repo_id='amathislab/mm-10m-2'))
    print(f'Teacher checkpoint: {teacher_root}')

    collection_cfg = BcCollectionConfig(
        samples_per_clip=175_000,   # 2 clips x 175k = 350k transitions total
        episode_len=400,
        seed=0,
    )
    dataset = collect_bc_dataset(teacher_root, model, list(clips.values()), collection_cfg)

    DATA_NPZ.parent.mkdir(parents=True, exist_ok=True)
    np.savez(
        DATA_NPZ,
        obs=dataset['obs'],
        actions=dataset['actions'],
        theta=dataset['theta'],
    )
    print(
        f'Saved {dataset["obs"].shape[0]} real teacher-rollout transitions to {DATA_NPZ} '
        f'(obs={dataset["obs"].shape}, actions={dataset["actions"].shape})'
    )


---
## 3 — Policy architecture

A 6-layer MLP with **SiLU activations and LayerNorm** residual blocks:

```
obs (528) -> Linear(512) -> SiLU
           -> [Linear -> LayerNorm -> SiLU] x 5
           -> actor_mean (354)   # muscle activations
           -> critic (1)         # value estimate (unused at test time)
```

Total parameters: ~1.8M.

In [ ]:
from myosuite.envs.myo.tasks.mimic.policy import ActorCritic

net = ActorCritic(obs_dim=OBS_DIM)  # 528-dim obs (+ 6 orientation features)
n_params = sum(p.numel() for p in net.parameters())
print(f'Parameters: {n_params:,}')
print(net)


---
## 4 - Training

Supervised MSE loss on the real 350k-transition dataset
(`runs/bc_directional_v2/bc_directional.npz`), 120 epochs, batch size 512,
Adam + cosine LR schedule (fewer epochs than the v1 run since the larger
dataset means far more gradient steps per epoch).

The best-validation-loss checkpoint is saved to
`runs/bc_directional_v2/policy_bc_best.pt` as a **flat `state_dict`**
(`torch.save(policy.state_dict(), CKPT)`) — loadable directly with
`model.load_state_dict(torch.load(CKPT))`, no wrapper-dict unwrap needed. An
earlier version of this cell saved `{'model_state_dict': ...}` while the
eval/render cells loaded the raw state dict directly, which crashed on load;
that mismatch is fixed here.

A real run of this cell (350k transitions, 120 epochs, CPU) took the
validation loss to 0.00767 (best, epoch 120) — modestly lower than v1's
0.0083, but see the Evaluation section below for the much larger gain this
produces in actual survival — see `runs/bc_directional_v2/train_history.json`
for the full curve.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

CKPT_DIR.mkdir(parents=True, exist_ok=True)

if CKPT.exists():
    print(f'Checkpoint already exists at {CKPT} — skipping training.')
else:
    d = np.load(DATA_NPZ)
    obs_t   = torch.as_tensor(d['obs'],     dtype=torch.float32)
    act_t   = torch.as_tensor(d['actions'], dtype=torch.float32)
    print(f'Dataset: {DATA_NPZ.name}  obs={tuple(obs_t.shape)}  act={tuple(act_t.shape)}')

    dataset = TensorDataset(obs_t, act_t)
    n_val   = max(1, int(0.05 * len(dataset)))
    n_train = len(dataset) - n_val
    train_ds, val_ds = random_split(dataset, [n_train, n_val])
    train_dl = DataLoader(train_ds, batch_size=512, shuffle=True,  num_workers=0)
    val_dl   = DataLoader(val_ds,   batch_size=512, shuffle=False, num_workers=0)

    policy = ActorCritic(obs_dim=OBS_DIM, act_dim=ACT_DIM)
    optimiser = optim.Adam(policy.parameters(), lr=3e-4)
    EPOCHS = 120
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=EPOCHS)
    loss_fn   = nn.MSELoss()

    train_losses, val_losses = [], []
    best_val = float('inf')
    for epoch in range(1, EPOCHS + 1):
        policy.train()
        running = 0.0
        for obs_b, act_b in train_dl:
            optimiser.zero_grad()
            pred, _ = policy(obs_b)
            loss = loss_fn(pred, act_b)
            loss.backward()
            optimiser.step()
            running += loss.item()
        scheduler.step()

        policy.eval()
        with torch.no_grad():
            val_loss = sum(
                loss_fn(policy(obs_b)[0], act_b).item()
                for obs_b, act_b in val_dl
            ) / len(val_dl)
        train_losses.append(running / len(train_dl))
        val_losses.append(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            # Flat state_dict -- the eval/render cells below load it directly
            # with `model.load_state_dict(torch.load(CKPT))`, no
            # `['model_state_dict']` unwrap.
            torch.save(policy.state_dict(), CKPT)

        if epoch % max(1, EPOCHS // 10) == 0:
            print(f'Epoch {epoch:4d}/{EPOCHS}  train={train_losses[-1]:.5f}  val={val_loss:.5f}')

    print(f'Best val loss: {best_val:.5f}  checkpoint: {CKPT}')


In [ ]:
# Plot loss curve (only if we just trained)
try:
    import matplotlib.pyplot as plt
    if 'train_losses' in dir():
        fig, ax = plt.subplots(figsize=(7, 3))
        ax.plot(train_losses, label='train')
        ax.plot(val_losses,   label='val')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('MSE loss')
        ax.legend()
        ax.set_title('BC training loss')
        plt.tight_layout()
        plt.show()
    else:
        print('Loss history not available (training was skipped).')
except ImportError:
    print('matplotlib not installed; skipping plot.')


---
## 5 — Evaluation

Load the best checkpoint and run the policy on `MuscleMimicFullbodyDirectionalEnv`
for each of the 8 compass directions, 10 seeds, 200 steps (~5 seconds).  
We report **survival@200** (fraction of runs that never fall within 200 steps),
**mean steps alive** (finer-grained than the binary survival flag), and
**alignment** (cosine similarity between displacement and commanded direction).

A real run of this cell against the 350k-transition `bc_directional_v2`
checkpoint — **the recommended default (see cell 2's fallback order)** —
gives: **survival@200 = 72.5%**, **mean steps alive = 190.6/200
(~95% of the episode)**, **mean alignment = +0.698**. This is well above
the original 60k-transition `bc_directional_v1` baseline (survival@200 =
6.2%, mean steps alive = 147.1/200, alignment = +0.746). A follow-up run
scaled further to 360k transitions (`bc_directional_v3`) was also tried,
but *regressed* to survival@200 = 50.0% despite a marginally lower
validation loss (0.00772 vs v2's 0.00767) — v3 is kept for comparison
only, not recommended as the default.

| Run | Transitions | Val loss | Survival@200 | Mean steps alive | Alignment |
|-----|-------------|----------|---------------|-------------------|-----------|
| v1 | 60,000 | 0.0083 | 6.2% | 147.1/200 | +0.746 |
| v2 | 350,000 | 0.00767 | **72.5%** | **190.6/200** | +0.698 |
| v3 | 360,000 | 0.00772 | 50.0% | 184.3/200 | +0.688 |

Per direction, v2 hits 100% survival on N/NW/W/SW, is strong on NE (90%),
but is weak on S (20%) and SE (0%). The v3 follow-up run showed a
*wider* weak set (NE/S/SE all at 0%, NW at 30%) despite more data — since
v1->v2->v3 used the same collection pipeline and only the sample count
changed, this looks like collection-seed variance in which regions of the
circular clips' state space get covered, rather than a saturating scaling
curve. `bc_directional_v2` is the best checkpoint by survival and the
recommended default; v3 is kept only for comparison.

This result strengthens the case for the DAgger correction pass raised in
the original v2 analysis: rather than further blind uniform-scaling of the
circular-clip collection (which has now shown diminishing and even
non-monotonic survival returns), a pass that rolls out the current best
student, relabels the states it actually visits with the teacher, and
retrains would directly target the compounding-error states causing the
weak-direction failures, instead of hoping more uniformly-sampled data
happens to cover them.

In [ ]:
from myosuite.envs.myo.tasks.mimic.cpu import MuscleMimicFullbodyDirectionalEnv

# Load policy
eval_policy = ActorCritic(obs_dim=OBS_DIM, act_dim=ACT_DIM)
ckpt_data = torch.load(CKPT, map_location='cpu', weights_only=True)
# Checkpoint is a flat state dict saved via torch.save(policy.state_dict(), ...)
eval_policy.load_state_dict(ckpt_data)
eval_policy.eval()
print(f'Loaded checkpoint from {CKPT}')

SECTOR_DEG_LIST = list(range(0, 360, 45))
SECTOR_LABELS   = ['E', 'NE', 'N', 'NW', 'W', 'SW', 'S', 'SE']
N_SEEDS  = 10
N_STEPS  = 200

results = {}
for deg, label in zip(SECTOR_DEG_LIST, SECTOR_LABELS):
    theta = math.radians(float(deg))
    survived, alignments, steps_alive = 0, [], []
    for seed in range(N_SEEDS):
        env = MuscleMimicFullbodyDirectionalEnv(
            seed=seed, target_speed=0.65,
            w_jpos=0.0, w_jvel=0.0, w_survival=0.3,
            angle_override=theta,
        )
        obs, _ = env.reset(seed=seed)
        start_xy = env.data.qpos[:2].copy()
        alive = True
        n_steps = 0
        for _ in range(N_STEPS):
            action = eval_policy.act(obs)
            obs, _, term, trunc, _ = env.step(action)
            n_steps += 1
            if term or trunc:
                alive = False
                break
        if alive:
            survived += 1
        steps_alive.append(n_steps)
        end_xy = env.data.qpos[:2].copy()
        disp = end_xy - start_xy
        norm = float(np.linalg.norm(disp))
        if norm > 1e-4:
            align = float((disp / norm) @ np.array([math.cos(theta), math.sin(theta)]))
        else:
            align = 0.0
        alignments.append(align)
        env.close()
    results[label] = {
        'surv': survived / N_SEEDS,
        'align': float(np.mean(alignments)),
        'mean_steps_alive': float(np.mean(steps_alive)),
    }

print(f'{'Dir':>4}  {'Surv@N':>7}  {'MeanSteps':>9}  {'Alignment':>9}')
for label, v in results.items():
    ok = 'OK' if v['surv'] >= 0.9 else '!'
    print(f'{ok} {label:>3}  {v["surv"]:>7.0%}  {v["mean_steps_alive"]:>9.1f}  {v["align"]:>+9.3f}')
overall_surv = float(np.mean([v['surv'] for v in results.values()]))
overall_steps = float(np.mean([v['mean_steps_alive'] for v in results.values()]))
overall_align = float(np.mean([v['align'] for v in results.values()]))
print(f'Overall survival@{N_STEPS}: {overall_surv:.1%}  '
      f'Mean steps alive: {overall_steps:.1f}/{N_STEPS}  '
      f'Overall alignment: {overall_align:+.3f}')


---
## 6 — Render: single agent, all 8 directions

One episode per direction (seed=2, 200 steps).  
Frames are stitched into a 4x2 grid video and shown inline with `mediapy`.

In [ ]:
import imageio

EIGHT_DIR_VIDEO = RENDER_DIR / 'bc_dagger_v1_8dirs.mp4'

if EIGHT_DIR_VIDEO.exists():
    print(f'Video already exists: {EIGHT_DIR_VIDEO}')
else:
    RENDER_W, RENDER_H = 320, 240
    FPS = 40

    grid_frames: list[np.ndarray] = []
    sector_frames: list[list[np.ndarray]] = [[] for _ in SECTOR_LABELS]

    for idx, (deg, label) in enumerate(zip(SECTOR_DEG_LIST, SECTOR_LABELS)):
        theta = math.radians(float(deg))
        env = MuscleMimicFullbodyDirectionalEnv(
            seed=2, target_speed=0.65,
            w_jpos=0.0, w_jvel=0.0, w_survival=0.3,
            angle_override=theta,
            render_mode='rgb_array',
        )
        obs, _ = env.reset(seed=2)
        renderer = mujoco.Renderer(env.model, height=RENDER_H, width=RENDER_W)
        # This model has no named 'track' camera -- use a fixed free camera
        # (same framing as the 1v1 render below).
        cam = mujoco.MjvCamera()
        cam.type = mujoco.mjtCamera.mjCAMERA_FREE
        cam.lookat[:] = [0.0, 0.0, 1.0]
        cam.distance = 3.5
        cam.azimuth = 135.0
        cam.elevation = -20.0

        for _ in range(N_STEPS):
            action = eval_policy.act(obs)
            obs, _, term, trunc, _ = env.step(action)
            renderer.update_scene(env.data, camera=cam)
            sector_frames[idx].append(renderer.render().copy())
            if term or trunc:
                break

        # Pad to N_STEPS with last frame
        while len(sector_frames[idx]) < N_STEPS:
            sector_frames[idx].append(sector_frames[idx][-1].copy())

        renderer.close()
        env.close()
        print(f'{label}: {len(sector_frames[idx])} frames')

    # Stitch into 4x2 grid
    for t in range(N_STEPS):
        row0 = np.concatenate([sector_frames[i][t] for i in range(4)], axis=1)
        row1 = np.concatenate([sector_frames[i][t] for i in range(4, 8)], axis=1)
        grid_frames.append(np.concatenate([row0, row1], axis=0).astype(np.uint8))

    with imageio.get_writer(str(EIGHT_DIR_VIDEO), fps=FPS, quality=8, macro_block_size=1) as w:
        for f in grid_frames:
            w.append_data(f)
    print(f'Saved: {EIGHT_DIR_VIDEO}')

print(f'Video: {EIGHT_DIR_VIDEO}')


---
## 7 - 1-vs-1 ChaseTag render

Uses `myoChallengeChaseTagFBVs-v0` directly (the registered multi-agent env —
this is the current id as of the "Consolidate chase-tag-vs 1v1 env_ids"
commit; earlier drafts of this notebook referenced the now-retired
`myoChallengeChaseTagVsFullbodySteered-v0` name, which was folded into this
single id).
Both agents run the same BC policy; per-agent **528-dim obs** are extracted from the
shared `MjData` via `_bc_obs_for_agent()` (local qpos/qvel + act + body-frame root vel + heading + orientation(6)).

**Initialisation:** circular gait clips matched by heading sector (no T-pose, no zero-activation fall).
`mj_resetData` leaves agents in a flat T-pose with zero muscle activation — they fall immediately.
The fix: copy qpos/qvel from a clip frame whose yaw matches the target heading, set `act=0.05`
(muscle activations must be reset per episode to avoid carryover from the previous episode).

**Tag termination fix:** `compute_damage` now respects each agent's CHASE/EVADE role — previously
it was hardcoded to treat agent_0 as the chaser regardless of the episode assignment, causing
incorrect termination signals.

**Headings:** opposite per episode so agents walk apart (no premature tag collision).

**Visual fixes:** per-agent floors hidden (z-fighting), agents tinted red/blue, arena walls added.

**Camera:** distance=5 m, azimuth=135 deg, elevation=-20 deg.

A real run with the 350k-transition `bc_directional_v2` checkpoint — the
recommended default (2 episodes, opposite headings,
`renders/chasetag_1v1/`) — produced two genuine `.mp4` renders: agent
survival times of 2.5s/3.6s (episode 0) and 1.4s/2.6s (episode 1) before
falling, well above the earlier `bc_directional_v1` checkpoint's 1.5s/1.8s
and 1.3s/2.5s. A follow-up run with the larger `bc_directional_v3`
checkpoint (`renders/chasetag_1v1_v3/`) gave a mixed result (2.0s/2.8s and
1.4s/3.9s) — not a uniform improvement, consistent with the CPU eval
section's finding that v3 is not recommended over v2. The 1v1 setting
remains harder than single-agent eval (agents can collide, and the policy
was never trained on the chase-tag arena/collision dynamics, only the
single-agent circular-clip rollouts).

In [ ]:
import gymnasium as gym
import torch
import imageio

import myosuite.envs.myo.tasks.challenge  # register myoChallengeChaseTagFBVs-v0

_TINT = 0.55


def _tint_agents(model):
    A0 = np.array([0.85, 0.15, 0.15, 1.0])
    A1 = np.array([0.15, 0.35, 0.85, 1.0])
    for i in range(model.ngeom):
        name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_GEOM, i) or ''
        orig = model.geom_rgba[i].copy()
        if orig[3] < 0.01:
            continue
        if name.startswith('a0_') and not name.endswith('_floor'):
            model.geom_rgba[i] = (1 - _TINT) * orig + _TINT * A0
            model.geom_rgba[i, 3] = orig[3]
        elif name.startswith('a1_') and not name.endswith('_floor'):
            model.geom_rgba[i] = (1 - _TINT) * orig + _TINT * A1
            model.geom_rgba[i, 3] = orig[3]


def _fix_floor(model):
    for i in range(model.ngeom):
        name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_GEOM, i) or ''
        if name in ('a0_floor', 'a1_floor'):
            model.geom_rgba[i, 3] = 0.0


def _add_arena_walls(spec):
    h = 4.5
    wb = spec.worldbody
    for name, pos, size in [
        ('arena_wall_n', [0.0,  h, 0.2], [h, 0.05, 0.2]),
        ('arena_wall_s', [0.0, -h, 0.2], [h, 0.05, 0.2]),
        ('arena_wall_e', [ h, 0.0, 0.2], [0.05, h, 0.2]),
        ('arena_wall_w', [-h, 0.0, 0.2], [0.05, h, 0.2]),
    ]:
        w = wb.add_geom()
        w.name = name
        w.type = mujoco.mjtGeom.mjGEOM_BOX
        w.size = np.array(size)
        w.pos = np.array(pos)
        w.rgba = np.array([1.0, 1.0, 0.0, 0.8])
        w.contype = 0
        w.conaffinity = 0
    new_model = spec.compile()
    return new_model, mujoco.MjData(new_model)


def _pelvis_yaw(qpos, adr):
    w, x, y, z = qpos[adr+3], qpos[adr+4], qpos[adr+5], qpos[adr+6]
    return float(np.arctan2(2*(w*z + x*y), 1 - 2*(y*y + z*z)))


def _bc_obs_for_agent(model, data, meta, agent_id, heading_dir):
    """Extract 528-dim obs for one agent from shared MjData.

    Uses contiguous qpos/qvel slices — identical to the single-agent env's
    ``data.qpos[7:]`` / ``data.qvel[6:]`` but offset to each agent's root.
    """
    jnt_ids = meta.jnt_ids[agent_id]
    root_qposadr = int(model.jnt_qposadr[jnt_ids[0]])   # free joint: 7 qpos DOFs
    root_qveladr = int(model.jnt_dofadr[jnt_ids[0]])    # free joint: 6 qvel DOFs
    local_qpos_start = int(model.jnt_qposadr[jnt_ids[1]])  # first hinge joint
    local_qvel_start = int(model.jnt_dofadr[jnt_ids[1]])
    nq_local = model.nq // 2 - 7   # 82 local joint DOFs per agent
    nv_local = model.nv // 2 - 6
    local_qpos = data.qpos[local_qpos_start : local_qpos_start + nq_local].astype(np.float32)
    local_qvel = data.qvel[local_qvel_start : local_qvel_start + nv_local].astype(np.float32)
    act = data.act[meta.act_indices[agent_id]].astype(np.float32)
    yaw = _pelvis_yaw(data.qpos, root_qposadr)
    vx, vy = data.qvel[root_qveladr], data.qvel[root_qveladr + 1]
    c, s = math.cos(-yaw), math.sin(-yaw)
    vel_body = np.array([c*vx - s*vy, s*vx + c*vy], dtype=np.float32)
    # 6-dim orientation: roll, pitch, wx_body, wy_body, wz_world, vz
    rq = data.qpos[root_qposadr + 3 : root_qposadr + 7]
    w, x, y, z = rq[0], rq[1], rq[2], rq[3]
    roll  = math.atan2(2*(w*x + y*z), 1 - 2*(x*x + y*y))
    pitch = math.asin(float(np.clip(2*(w*y - z*x), -1.0, 1.0)))
    wx_w  = data.qvel[root_qveladr + 3]
    wy_w  = data.qvel[root_qveladr + 4]
    wz_w  = data.qvel[root_qveladr + 5]
    wx_b  = c * wx_w - s * wy_w
    wy_b  = s * wx_w + c * wy_w
    vz    = data.qvel[root_qveladr + 2]
    orientation = np.array([roll, pitch, wx_b, wy_b, wz_w, vz], dtype=np.float32)
    return np.concatenate([local_qpos, local_qvel, act, vel_body,
                           heading_dir.astype(np.float32), orientation])


def _rotate_quat(qpos_slice, yaw_delta):
    half = yaw_delta / 2.0
    rw, rz = math.cos(half), math.sin(half)
    ow, ox, oy, oz = qpos_slice[3], qpos_slice[4], qpos_slice[5], qpos_slice[6]
    qpos_slice[3] = rw*ow - rz*oz
    qpos_slice[4] = rw*ox + rz*oy
    qpos_slice[5] = rw*oy - rz*ox
    qpos_slice[6] = rw*oz + rz*ow




# Load circular clips directly from HuggingFace (same two clips used for
# teacher-rollout data collection in Section 2 -- CCW08, not CCW04, matches
# the E/SW sector assignment computed in Section 1).
from pathlib import Path
from myosuite.core.trajectory_io import load_motion_clip
from huggingface_hub import hf_hub_download

_CLIP_FILES = [
    ("amathislab/musclemimic-retargeted", "MyoFullBody/gmr/KIT/4/WalkInClockwiseCircle01_poses.npz"),
    ("amathislab/musclemimic-retargeted", "MyoFullBody/gmr/KIT/4/WalkInCounterClockwiseCircle08_poses.npz"),
]

_ref_env = gym.make("myoFullBodyDirectional-v0", max_episode_steps=10)
_nq, _nv = _ref_env.unwrapped.model.nq, _ref_env.unwrapped.model.nv
_ref_env.close()

circ_clips = []
for _repo, _fname in _CLIP_FILES:
    _p = Path(hf_hub_download(repo_id=_repo, filename=_fname, repo_type="dataset"))
    circ_clips.append(load_motion_clip(_p, expected_nq=_nq, expected_nv=_nv))
print(f"Loaded {len(circ_clips)} circular clips.")


In [ ]:
# ── 1-vs-1 render ──────────────────────────────────────────────────────────

CHASETAG_DIR = RENDER_DIR / 'chasetag_1v1'
CHASETAG_DIR.mkdir(exist_ok=True)

N_EPS    = 2   # 2 episodes keeps this cell's runtime modest; increase for more coverage
FPS_1V1  = 50   # 4x slow-motion (sim at 200 Hz)
RENDER_W = 640
RENDER_H = 480

_TRAINED       = np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315])
_EP_HEADINGS   = [_TRAINED[i % 8] for i in range(N_EPS)]

policy_1v1 = ActorCritic(obs_dim=OBS_DIM, act_dim=ACT_DIM)
policy_1v1.load_state_dict(torch.load(str(CKPT), map_location='cpu', weights_only=True))
policy_1v1.eval()

env_1v1 = gym.make('myoChallengeChaseTagFBVs-v0')
raw = env_1v1.unwrapped
meta = raw._meta
raw._config.fall_pelvis_z_threshold = 0.6
_fix_floor(raw.model)
_tint_agents(raw.model)
raw.model, raw.data = _add_arena_walls(raw._spec)
_fix_floor(raw.model)
_tint_agents(raw.model)

renderer_1v1 = mujoco.Renderer(raw.model, height=RENDER_H, width=RENDER_W)
rng_1v1 = np.random.default_rng(42)

for ep in range(N_EPS):
    env_1v1.reset(seed=ep)
    h0 = _EP_HEADINGS[ep]
    h1 = (h0 + math.pi) % (2 * math.pi)
    # Initialize from best-matching clip frame (no quaternion rotation — stable)
    headings_used = {}
    for aid, angle in [('agent_0', h0), ('agent_1', h1)]:
        jnt_ids = meta.jnt_ids[aid]
        rqa = int(raw.model.jnt_qposadr[jnt_ids[0]])
        rva = int(raw.model.jnt_dofadr[jnt_ids[0]])
        root_xy = raw.data.qpos[rqa:rqa+2].copy()
        best_clip, best_fi, best_err = None, 0, float('inf')
        for clip in circ_clips:
            for fi in range(len(clip.qpos)):
                yaw_f = float(np.arctan2(
                    2*(clip.qpos[fi,3]*clip.qpos[fi,6] + clip.qpos[fi,4]*clip.qpos[fi,5]),
                    1-2*(clip.qpos[fi,5]**2+clip.qpos[fi,6]**2)))
                err = abs(math.atan2(math.sin(yaw_f-angle), math.cos(yaw_f-angle)))
                if err < best_err:
                    best_err, best_clip, best_fi = err, clip, fi
        qp, qv = best_clip.qpos[best_fi], best_clip.qvel[best_fi]
        actual_yaw = float(np.arctan2(2*(qp[3]*qp[6]+qp[4]*qp[5]),1-2*(qp[5]**2+qp[6]**2)))
        for k, jid in enumerate(jnt_ids[1:]):
            raw.data.qpos[int(raw.model.jnt_qposadr[jid])] = qp[7+k]
            raw.data.qvel[int(raw.model.jnt_dofadr[jid])]  = qv[6+k]
        raw.data.qpos[rqa:rqa+2] = root_xy
        raw.data.qpos[rqa+2] = qp[2]
        raw.data.qpos[rqa+3:rqa+7] = qp[3:7]
        raw.data.qvel[rva:rva+6] = qv[0:6]
        # Reset activations to 0.05 to avoid carryover from prior episode
        raw.data.act[meta.act_indices[aid]] = 0.05
        headings_used[aid] = np.array([math.cos(actual_yaw), math.sin(actual_yaw)], np.float32)
    mujoco.mj_forward(raw.model, raw.data)

    frames = []
    FALL_Z = 0.6
    fell = {'agent_0': False, 'agent_1': False}
    fall_step = {'agent_0': None, 'agent_1': None}
    for step in range(2000):
        for aid in ('agent_0', 'agent_1'):
            if not fell[aid]:
                obs = _bc_obs_for_agent(raw.model, raw.data, meta, aid, headings_used[aid])
                with torch.no_grad():
                    mu, _ = policy_1v1(torch.tensor(obs, dtype=torch.float32).unsqueeze(0))
                raw.data.ctrl[meta.act_indices[aid]] = np.clip(
                    mu.squeeze(0).numpy(), 0.0, 1.0)
            else:
                raw.data.ctrl[meta.act_indices[aid]] = 0.0
        for _ in range(5):
            mujoco.mj_step(raw.model, raw.data)
        for aid in ('agent_0', 'agent_1'):
            jids = meta.jnt_ids[aid]
            pz = raw.data.qpos[int(raw.model.jnt_qposadr[jids[0]])+2]
            if not fell[aid] and pz < FALL_Z:
                fell[aid] = True; fall_step[aid] = step + 1
        cam = mujoco.MjvCamera()
        cam.type = mujoco.mjtCamera.mjCAMERA_FREE
        cam.lookat[:] = [0.0, 0.0, 1.0]
        cam.distance = 5.0
        cam.azimuth = 135.0
        cam.elevation = -20.0
        renderer_1v1.update_scene(raw.data, camera=cam)
        frames.append(np.array(renderer_1v1.render(), dtype=np.uint8))
        if all(fell.values()):
            break
    out = CHASETAG_DIR / f'ep{ep:02d}_steps{step}.mp4'
    with imageio.get_writer(str(out), fps=FPS_1V1, quality=8, macro_block_size=1) as w:
        for f in frames:
            w.append_data(f)
    a0s = fall_step['agent_0'] or len(frames)
    a1s = fall_step['agent_1'] or len(frames)
    print(f'ep{ep:02d}: a0={a0s}steps({a0s/100:.1f}s) a1={a1s}steps({a1s/100:.1f}s) -> {out.name}')

renderer_1v1.close()
env_1v1.close()


---
## Summary

| Step | What we did | Output |
|------|-------------|--------|
| 1 - Clip selection | Circular clips (KIT/4 CW+CCW) cover all directions | `sector_assignments` dict |
| 2 - Data collection | Real teacher rollouts: 175k x 2 circular clips = 350,000 transitions (**recommended default**; scaled up from v1's 60k; still short of the original 800k design) | `runs/bc_directional_v2/bc_directional.npz` |
| 3 - Architecture | 6-layer SiLU+LN MLP, **528-dim obs** (qpos+qvel+act+vel\_body+heading+orientation) | `ActorCritic` class |
| 4 - Training | MSE on 350k real transitions, 120 epochs, cosine LR; val loss → 0.00767 (best) | `runs/bc_directional_v2/policy_bc_best.pt` |
| 5 - Evaluation (single-agent, 8 dir) | 10 seeds x 200 steps/dir; survival@200=72.5%, mean steps alive=190.6/200 (~95%), mean alignment=+0.698 (best of v1/v2/v3, see below) | `runs/bc_directional_v2/eval_results.json` |
| 6 - 8-dir render | Tracking camera, 4x2 grid | `renders/bc_dagger_v1_8dirs.mp4` |
| 7 - 1-vs-1 ChaseTag | `myoChallengeChaseTagFBVs-v0`, 528-dim obs per agent; 2 real episodes, agent survival 1.4-3.6s before falling | `renders/chasetag_1v1/ep*.mp4` |

**Key lessons:**
- Circular clips sweep all directions continuously — no per-sector data collection needed; the clip's own instantaneous walking direction becomes the `heading_cmd` label for free.
- Loading the teacher's Orbax checkpoint needs `orbax-checkpoint`, which is excluded on macOS in this repo's `pyproject.toml` (`mjx` extra's `sys_platform != 'darwin'` markers) — real data collection has to run on Linux (we used Modal; see `scripts/modal_bc_directional_pipeline.py`).
- Checkpoint save/load must agree on format: saving `{'model_state_dict': ...}` while loading a raw `state_dict` directly crashes — this notebook now saves a flat `state_dict` everywhere.
- Scaling the BC dataset from 60k to 350k transitions (5.8x, still only 44% of the original 800k design target) took survival@200 from 6.2% to 72.5% and mean episode survival from ~74% to ~95% — this confirms BC's classic covariate-shift failure mode (the policy drifting into states the teacher never demonstrated) was a major limiter.
- Scaling further from 350k to 360k transitions (v2 -> v3) did **not** repeat that gain: v3's validation loss is essentially tied with v2's (0.00772 vs 0.00767) but its survival@200 *regressed* to 50.0% from 72.5%, with a wider set of weak directions (NE/S/SE all at 0%, NW at 30%, vs. v2's S at 20%/SE at 0% only). Since the collection pipeline and sample count per clip are the only things that changed between v2 and v3, this looks like collection-seed variance in which parts of the circular clips' state space get sampled, not a reproducible scaling law — i.e. more uniformly-collected data does not reliably buy more uniform heading coverage once you're past the initial covariate-shift regime.
- Validation loss (single-step action MSE) is a weak proxy for closed-loop survival once BC has already closed most of the covariate-shift gap — v2 and v3 have nearly identical val loss but a 22.5-point survival@200 gap, so val loss alone should not be used to pick a checkpoint for deployment; always confirm with the closed-loop eval.
- `bc_directional_v2` (350k transitions) remains the best checkpoint by survival@200 and is recommended as the default; `bc_directional_v3` (360k) is kept for comparison and has the lowest validation loss of the three runs, but is not currently the recommended default.
- A 60k-transition BC dataset (7.5% of the original 800k design target) is enough to produce a genuinely functional — if not yet robust — directional policy: strong heading alignment (+0.75 mean, up to +0.96) and ~74% mean episode survival, dramatically better than random, but not yet matching the original design's presumed larger-dataset robustness.
- Body-frame root velocity makes heading-invariant gait easier to learn.
- Adding 6-dim orientation features (roll, pitch, angular vel) improves tilt recovery.
- `heading_cmd` is at `obs[520:522]` — after `vel_body` at `[518:520]`. Using `[518:520]` for heading confuses velocity and command signals.
- `data.act[:] = 0.05` must be set at every episode reset — leftover activations from the prior episode cause immediate falls.
- Tag termination must respect each agent's assigned CHASE/EVADE role, not hardcode agent_0 as chaser.
- 528-dim obs transfers directly to 1v1: extract per-agent slice from shared `MjData`.
- Avoid z-fighting by recolouring the existing floor geom, not stacking a second plane.
- Opposite headings per episode prevent premature tag collisions in the render.
- `myoChallengeChaseTagFBVs-v0` is the current 1v1 env id — an earlier `myoChallengeChaseTagVsFullbodySteered-v0` name was consolidated into it and no longer exists.

**Next steps to close the remaining survival gap:** the dataset-scaling
experiments above (60k -> 350k -> 360k transitions) show that most of the
gain came from the first jump (60k -> 350k), and the second jump (350k ->
360k) produced no reliable further improvement -- and a regression in this
particular run. Blindly collecting more uniformly-sampled transitions
toward the original 800k target is therefore not obviously the best next
move. A DAgger correction pass (roll out the `bc_directional_v2` student --
the current best checkpoint -- relabel the states it actually visits with
the teacher, and retrain) targeting the weak directions directly addresses
the compounding-error mechanism that uniform circular-clip collection
cannot fix, and is likely more sample-efficient than further scaling.


## 8 - Chase-tag observation extension (FBP2)

The directional-locomotion policy above (`bc_directional_v2`) has no opponent awareness -- it was trained purely on single-agent proprioception + a heading command. To use it in a chase-tag setting, the observation can be extended **additively**: the same 528-dim locomotion block stays byte-identical as a leading prefix, with a 7-dim opponent-relative block (relative position, relative velocity, distance) and a 2-dim chaser/runner role one-hot appended after it -- 537 dims total. See `myosuite/envs/myo/tasks/mimic/chasetag_obs.py`.

This additive design means `bc_directional_v2`'s pretrained weights can seed a larger network directly: `ActorCritic.load_expanded(ckpt, new_obs_dim=537, ...)` (`myosuite/envs/myo/tasks/mimic/policy.py`) copies the pretrained first-layer weights into the leading 528 input columns. The new 9 columns are **zeroed** rather than left at random init -- those raw opponent-relative features have real-world magnitude (meters, m/s), and random weights on them injected large noise into the trunk, corrupting the warm-started policy's behavior before any fine-tuning even started. Zeroing makes the expanded network exactly reproduce the pretrained policy on the shared 528 dims.

**Task**: `myoChallengeChaseTagFBP2-v0` -- a full-body (354-muscle) agent facing a scripted, passively-wandering opponent, registered on both CPU (`ChaseTagEnv`) and GPU/mjlab (same env_id, per this repo's cross-backend contract).

### Results (30 seeds x 200 steps, CPU eval)

| Policy | Mean steps alive | Fall rate |
|---|---|---|
| `bc_directional_v2` via `load_expanded` (corrected warm-start, no fine-tuning) | **147.2** | 93.3% |

A PPO fine-tuning pass on top of this warm-start was attempted (a compact from-scratch PPO loop driving mjlab's `ManagerBasedRlEnv` directly, warm-started via `load_expanded` with `log_std` reset for stability). Across 300 iterations the training reward stayed flat to slightly negative with no improving trend, and the resulting policy performed substantially worse than the untouched warm-start (66.3 mean steps alive vs. 147.2). The chase-tag reward signal used (dominated by distance-to-opponent against a mostly-stationary opponent) does not appear to provide a useful gradient toward staying upright, and the fine-tuning pipeline has been removed pending a reward redesign (e.g. an explicit anti-fall term or a KL penalty back to the BC prior). Until then, the plain warm-started directional policy is the better choice for this task.

**Not covered here**: `myoChallengeChaseTagFBVs-v0` (1v1 self-play) has no GPU/mjlab registration -- mjlab has no multi-agent/self-play support anywhere in this codebase, so GPU self-play remains a separate, larger future project.